# LEXICON-X · Expedición Eráviri — Gramática inducida frente a recuperación

**Pregunta de investigación:** ¿qué estrategia generaliza mejor a combinaciones nuevas cuando las traducciones paralelas son escasas (420 pares)?

**Resumen de resultados (validación, semilla 20261014):**

| Sistema | Exact Match |
|---|---:|
| Baseline Jaccard (recuperación) | 0.000 |
| **Gramática por slots + morfología compuesta (propuesta)** | **1.000** |

El resultado más informativo no es el 1.000 de validación, sino el experimento de generalización (OOD) y la curva de datos de la sección 5, donde se ve qué componente aporta y dónde falla el método.

**Hipótesis falsables**

- **H1 (composicionalidad):** Eráviri sigue la plantilla `Sujeto Objeto [Locativo] [nu] Verbo(+sufijo)`, y cada slot se puede traducir de forma independiente. *Se falsa si* un método que solo recupera frases completas iguala a uno que compone por slots.
- **H2 (morfología):** el sufijo verbal es productivo (`-ta` pasado, `-ri` futuro, presente sin sufijo). *Se falsa si* segmentar el sufijo no mejora la generalización a combinaciones verbo×tiempo no vistas.

**Estructura:** reproducibilidad → carga → EDA → baseline → método → ablaciones → errores → exportación → Fase 5 → conclusión.

## 0. Reproducibilidad

**Decisión:** el método no usa redes neuronales ni aleatoriedad, así que el resultado final es determinista. Aun así fijamos `random.seed` y `PYTHONHASHSEED`, porque el muestreo de la curva de datos (celda 5) sí usa un generador aleatorio, y cada submuestra usa su propia semilla (0 a 19) para que sea repetible.

**Dependencias:** solo la librería estándar de Python. `sacrebleu` es opcional y solo se usa para chrF.

In [1]:
import os, random, csv, re, collections, platform, sys
SEED = 20261014
os.environ['PYTHONHASHSEED'] = str(SEED); random.seed(SEED)
print(sys.version.split()[0], platform.platform(), '| semilla', SEED, '| dependencias: solo librería estándar (+sacrebleu opcional)')

3.14.8 Windows-11-10.0.26200-SP0 | semilla 20261014 | dependencias: solo librería estándar (+sacrebleu opcional)


## 1. Carga de datos

Se cargan los cuatro archivos oficiales. La función `norm` (minúsculas, sin puntuación, espacios normalizados) se usa **en todas las métricas, tanto en predicciones como en referencias**, para que la comparación sea consistente con la definición de Exact Match del enunciado.

In [2]:
from pathlib import Path
BASE = Path('/content/datos') if Path('/content/datos/train_parallel.csv').exists() else Path('datos')
def load_csv(n):
    with (BASE/n).open(encoding='utf-8', newline='') as f: return list(csv.DictReader(f))
train, val = load_csv('train_parallel.csv'), load_csv('validation_parallel.csv')
mono, public = load_csv('corpus_monolingual.csv'), load_csv('test_public_unlabeled.csv')
toks = lambda s: re.findall(r"[a-záéíóúñü]+", s.lower())
def norm(s):
    s = re.sub(r'[^\w\sáéíóúñü]', '', s.lower().strip()); return ' '.join(s.split())
def exact_match(rows, f): return sum(norm(f(r['naviri'])) == norm(r['espanol']) for r in rows) / len(rows)

## 2. Exploración e hipótesis

Antes de elegir método se inspeccionó la estructura de la lengua. Las observaciones que sostienen H1 y H2 son:

1. **Posiciones con vocabulario cerrado.** Cada posición tiene pocos tipos: la posición 0 tiene 10 (los sujetos), la 1 tiene 12 (los objetos) y las posiciones finales contienen locativos, `nu` y verbos.
2. **Vocabulario muy pequeño.** El corpus monolingüe tiene **59 tipos**, y validación y test público tienen **0 palabras fuera del vocabulario de train**.
3. **Sufijos productivos.** Las palabras verbales aparecen como raíz sola, raíz+`ta` y raíz+`ri`.
4. **Ninguna frase de validación o test está en train.** Por eso un método que solo copia traducciones completas no puede acertar (Exact Match = 0 en el baseline).

La observación 4 es lo que hace falsable H1: si la lengua no fuera composicional, el baseline no habría fallado de forma tan sistemática.

In [3]:
print('Longitudes (tokens):', collections.Counter(len(toks(r['naviri'])) for r in train))
for i in range(5):
    c = collections.Counter(toks(r['naviri'])[i] for r in mono if len(toks(r['naviri'])) > i)
    print(f'pos {i}: {len(c)} tipos | {c.most_common(6)}')
V = collections.Counter(w for r in mono for w in toks(r['naviri']))
print('Vocabulario monolingüe:', len(V), 'tipos')
print('OOV val/test respecto a train:',
      sum(w not in {t for r in train for t in toks(r['naviri'])} for r in val for w in toks(r['naviri'])),
      sum(w not in {t for r in train for t in toks(r['naviri'])} for r in public for w in toks(r['naviri'])))

Longitudes (tokens): Counter({4: 269, 3: 95, 5: 56})
pos 0: 10 tipos | [('mika', 319), ('sari', 310), ('raku', 309), ('naku', 300), ('penu', 299), ('telu', 297)]
pos 1: 12 tipos | [('tavi', 268), ('beko', 267), ('tora', 260), ('feni', 253), ('rima', 251), ('nelo', 249)]
pos 2: 37 tipos | [('rilo', 375), ('kuna', 354), ('sova', 353), ('hala', 341), ('pavo', 335), ('miri', 331)]
pos 3: 31 tipos | [('nu', 399), ('faro', 133), ('lomu', 126), ('dema', 123), ('nemi', 122), ('paki', 119)]
pos 4: 30 tipos | [('yelo', 30), ('taku', 29), ('dema', 28), ('sema', 28), ('lomu', 28), ('vasi', 27)]
Vocabulario monolingüe: 59 tipos
OOV val/test respecto a train: 0 0


## 3. Baseline: recuperación por similitud de Jaccard

El baseline obligatorio copia la traducción del ejemplo de entrenamiento con más tokens en común. Da **EM = 0.000**.

**Por qué es esperable:** no hay ninguna frase de validación en train, y la recuperación solo puede devolver frases completas ya vistas. No es un baseline "mal hecho": es la prueba de que esta tarea exige generalización composicional y no memorización.

In [4]:
def jaccard(a, b):
    a, b = set(toks(a)), set(toks(b)); return len(a & b) / len(a | b) if a | b else 0.0
def baseline_translate(s): return max(train, key=lambda r: jaccard(s, r['naviri']))['espanol']
print('Baseline EM val:', round(exact_match(val, baseline_translate), 4))

Baseline EM val: 0.0


## 4. Método propuesto: segmentación morfológica + parseo por slots + léxico inducido

**Decisiones y razones**

- **Sin BPE ni modelos preentrenados.** Con 59 tipos de palabra no hay subpalabras que fusionar, y no existe un modelo preentrenado para una lengua sintética. La segmentación relevante es lingüística, no estadística: `raíz + {∅, -ta, -ri}`.
- **Los stems verbales se descubren con el corpus monolingüe**, sin etiquetas ni español: una palabra `w` es "raíz+sufijo" si `w[:-2]` también aparece en posición final. Es la única parte del corpus mono que usa el método.
- **Parseo por slots** (`S, O, LOC, NEG, V, T`). `nu` se trata como marca de negación y no como palabra de contenido; el locativo existe solo si la frase tiene 4 palabras sin contar `nu`.
- **Léxico inducido por slot desde los pares paralelos.** El artículo viaja con el sustantivo (`el viajero`, `la casa`), porque su género depende del sustantivo y no de la posición.
- **Inflexión española inducida de datos.** Para combinaciones verbo×tiempo no vistas, se aprende una regla de sustitución final (última letra del presente → terminación) a partir de los pares vistos.
- **No se afinó nada contra validación.** El léxico y las reglas salen solo de train; validación se usa solo para comparar decisiones.

**Por qué debería generalizar:** los parámetros son tablas de léxico por slot más una regla morfológica, no pesos que se ajusten al ruido; con tan pocos datos eso reduce el riesgo de sobreajuste.

In [5]:
# Los stems verbales se descubren SOLO con distribución (corpus mono): w es "stem+sufijo" si w[:-2] también ocurre
# en posición final. El corpus mono NO usa etiquetas ni español.
def discover_stems(corpus_rows):
    last = collections.Counter(toks(r['naviri'])[-1] for r in corpus_rows)
    stems = {w for w in last if not (w.endswith(('ta', 'ri')) and w[:-2] in last)}
    return stems
SUFFIX = {'ta': 'PAST', 'ri': 'FUT'}
def split_verb(w, stems):
    for suf, t in SUFFIX.items():
        if w.endswith(suf) and w[:-2] in stems: return w[:-2], t
    return w, 'PRES'
def parse_es(e):  # plantilla del español: SN [no] V SN [en SN]
    return re.match(r'^((?:el|la) \w+) (no )?(\w+) ((?:el|la) \w+)(?: en ((?:el|la) \w+))?$', norm(e))
def parse_nv(s, stems):
    toks_ = toks(s); neg = 'nu' in toks_; t = [w for w in toks_ if w != 'nu']
    stem, tense = split_verb(t[-1], stems)
    return dict(S=t[0], O=t[1], LOC=(t[2] if len(t) == 4 else None), NEG=neg, V=stem, T=tense)

class SlotTranslator:
    def __init__(self, atomic_verbs=False, use_mono=True, mono_rows=None):
        self.atomic, self.use_mono = atomic_verbs, use_mono
        self.mono = mono_rows if mono_rows is not None else mono
    def fit(self, rows):
        corpus = self.mono if self.use_mono else rows  # ABLACIÓN: sin mono, los stems salen solo de train
        self.stems = discover_stems(corpus if self.use_mono else [{'naviri': r['naviri']} for r in rows])
        lex = collections.defaultdict(collections.Counter)
        for r in rows:
            m = parse_es(r['espanol']); p = parse_nv(r['naviri'], self.stems)
            if not m: continue
            lex[('S', p['S'])][m.group(1)] += 1; lex[('O', p['O'])][m.group(4)] += 1
            if p['LOC']: lex[('L', p['LOC'])][m.group(5)] += 1
            lex[('V', p['V'], p['T'])][m.group(3)] += 1
        self.lex = {k: v.most_common(1)[0][0] for k, v in lex.items()}
        self.pres = {k[1]: f for k, f in self.lex.items() if k[0] == 'V' and k[2] == 'PRES'}
        # regla de inflexión española inducida: (última letra del presente) -> reemplazo, por tiempo
        self.rule = {'PAST': collections.Counter(), 'FUT': collections.Counter()}
        for k, f in self.lex.items():
            if k[0] == 'V' and k[2] != 'PRES' and k[1] in self.pres:
                p = self.pres[k[1]]; self.rule[k[2]][(p[-1], f[len(p) - 1:])] += 1
        return self
    def verb(self, stem, t):
        if ('V', stem, t) in self.lex: return self.lex[('V', stem, t)]
        if self.atomic or stem not in self.pres: return '?'
        p = self.pres[stem]
        for (last, rep), _ in self.rule[t].most_common():
            if p[-1] == last: return p[:-1] + rep
        return p
    def __call__(self, s):
        p = parse_nv(s, self.stems)
        out = f"{self.lex.get(('S', p['S']), '?')} {'no ' if p['NEG'] else ''}{self.verb(p['V'], p['T'])} {self.lex.get(('O', p['O']), '?')}"
        if p['LOC']: out += f" en {self.lex.get(('L', p['LOC']), '?')}"
        return out + '.'

model = SlotTranslator().fit(train)
def my_translate(s): return model(s)
print('Mi sistema EM val:', round(exact_match(val, my_translate), 4))

Mi sistema EM val: 1.0


## 5. Resultados experimentales y ablaciones

**Resultados obtenidos (semilla 20261014)**

| Experimento | EM |
|---|---:|
| Baseline Jaccard | 0.000 |
| Método completo | **1.000** |
| A1: sin corpus monolingüe (stems solo de train) | 1.000 |
| OOD verbo×tiempo, morfología compuesta | 0.500 |
| OOD verbo×tiempo, verbos atómicos (sin segmentar) | 0.000 |

**Curva de datos** (media de 20 submuestras aleatorias de train):

| Pares de entrenamiento | EM medio | EM mínimo |
|---:|---:|---:|
| 10 | 0.124 | 0.075 |
| 20 | 0.460 | 0.292 |
| 40 | 0.851 | 0.658 |
| 80 | 0.970 | 0.900 |
| 160 | 0.993 | 0.958 |
| 420 | 1.000 | 1.000 |

**Interpretación**

- **H1 se sostiene.** El método por slots pasa de 0.000 a 1.000 frente al baseline, y con solo 80 pares ya llega a ~0.97. Un enfoque de recuperación nunca ve suficientes combinaciones; uno composicional las genera.
- **H2 se sostiene.** En el experimento OOD, se retiraron del entrenamiento cuatro combinaciones verbo×tiempo (`dema`+pasado, `faro`+futuro, `nemi`+pasado, `lomu`+futuro) y se evaluó solo sobre las 10 frases de validación que las usan. Segmentar el sufijo da **0.5** y tratar cada forma verbal como palabra atómica da **0.0**. El efecto de la morfología compuesta es de 0.5 puntos de EM, con un tamaño de muestra pequeño (n = 10).
- **A1: el corpus monolingüe no mejoró la métrica.** Sin él el resultado es el mismo (1.000), porque train ya contiene todas las raíces verbales. Es un resultado negativo honesto: en esta lengua sintética, el corpus mono sirvió para **confirmar** la estructura (59 tipos, 100 % de frases dentro de la plantilla, raíces verbales) pero no para mejorar la traducción. Con menos datos de train, su valor podría ser mayor, y eso queda como experimento futuro.
- **La curva de datos muestra la eficiencia** del enfoque: bajo ~40 pares el rendimiento es inestable (el mínimo es 0.075 con 10 pares), porque aún faltan palabras o combinaciones en el léxico.

In [6]:
def subset_ood(rows, hold):  # retiene combinaciones (verbo, tiempo) del entrenamiento
    st = model.stems
    f = lambda r: (lambda p: (p['V'], p['T']))(parse_nv(r['naviri'], st))
    return [r for r in rows if f(r) not in hold], [r for r in val if f(r) in hold]
hold = {('dema', 'PAST'), ('faro', 'FUT'), ('nemi', 'PAST'), ('lomu', 'FUT')}
tr_h, va_h = subset_ood(train, hold)
res = {
 'baseline Jaccard': exact_match(val, baseline_translate),
 'método completo': exact_match(val, my_translate),
 'A1 sin corpus mono (stems solo de train)': exact_match(val, SlotTranslator(use_mono=False).fit(train)),
 'OOD verbo×tiempo | morfología compuesta': exact_match(va_h, SlotTranslator().fit(tr_h)),
 'OOD verbo×tiempo | verbos atómicos (sin segmentar)': exact_match(va_h, SlotTranslator(atomic_verbs=True).fit(tr_h)),
}
for k, v in res.items(): print(f'{v:6.3f}  {k}')
print('Curva de datos (media de 20 semillas):')
for n in (10, 20, 40, 80, 160, 420):
    accs = []
    for sd in range(20):
        rnd = random.Random(sd); sub = rnd.sample(train, n)
        accs.append(exact_match(val, SlotTranslator().fit(sub)))
    print(f'  n={n:3d}  EM={sum(accs)/20:.3f}  min={min(accs):.3f}')

 0.000  baseline Jaccard
 1.000  método completo
 1.000  A1 sin corpus mono (stems solo de train)
 0.500  OOD verbo×tiempo | morfología compuesta
 0.000  OOD verbo×tiempo | verbos atómicos (sin segmentar)
Curva de datos (media de 20 semillas):
  n= 10  EM=0.124  min=0.075
  n= 20  EM=0.460  min=0.292
  n= 40  EM=0.851  min=0.658
  n= 80  EM=0.970  min=0.900
  n=160  EM=0.993  min=0.958
  n=420  EM=1.000  min=1.000


## 6. Análisis de errores

La validación estándar no tiene errores (EM = 1.0), así que los errores se analizan sobre el **subconjunto OOD**, donde el método fallaba 5 de 10 frases. **Los cinco errores comparten la misma causa**, y tres de ellos bastan para ilustrarla:

| # | Eráviri | Referencia | Predicción |
|---|---|---|---|
| 1 | `lina nelo hala nu nemita` | la artista no **recordó** el mensaje en la casa | la artista no **recuerdó** el mensaje en la casa |
| 2 | `yami dali nemita` | la estudiante **recordó** la fruta | la estudiante **recuerdó** la fruta |
| 3 | `kemi suni sova nemita` | la investigadora **recordó** el libro en la escuela | la investigadora **recuerdó** el libro en la escuela |

**Diagnóstico.** El error está en la capa de **morfología del español**, no en Eráviri: la segmentación `nemi + -ta` es correcta, la raíz se traduce bien (`recuerda`) y el orden, el sujeto, el objeto y el locativo son correctos. Lo que falla es la regla inducida: aprende "reemplazar la `a` final por `ó`" (como en `lleva → llevó`), pero `recordar` tiene cambio de raíz (`recuerda → recordó`, no `recuerdó`). Una regla de sufijo no puede capturar un cambio de vocal en medio de la palabra.

**Implicación para chrF.** El error es de un solo carácter (`recuerdó` frente a `recordó`), así que chrF lo penaliza poco, pero el Exact Match lo cuenta como frase completamente mal. Por eso conviene reportar ambas métricas.

**Mejora posible:** añadir un diccionario de verbos irregulares o aprender alternancias vocálicas (`ue ↔ o`) como una segunda regla; para ello se necesita ver al menos un par irregular por clase en train.

**Nota sobre chrF.** En validación estándar el chrF da 100.00, porque el EM es 1.0 y por tanto no discrimina. El análisis por fenómeno y el experimento OOD son donde las métricas muestran diferencias.

In [7]:
m_h = SlotTranslator().fit(tr_h)
errs = [(r['naviri'], r['espanol'], m_h(r['naviri'])) for r in va_h if norm(m_h(r['naviri'])) != norm(r['espanol'])]
for s, ref, pred in errs[:5]: print('NÁVIRI:', s, '\nREF:   ', ref, '\nPRED:  ', pred, '\n')
print('Errores OOD:', len(errs), '/', len(va_h))
# chrF opcional: !pip -q install sacrebleu
try:
    from sacrebleu.metrics import CHRF
    print('chrF val:', CHRF().corpus_score([my_translate(r['naviri']) for r in val], [[r['espanol'] for r in val]]))
except ImportError: pass

NÁVIRI: lina nelo hala nu nemita. 
REF:    la artista no recordó el mensaje en la casa. 
PRED:   la artista no recuerdó el mensaje en la casa. 

NÁVIRI: yami dali nemita. 
REF:    la estudiante recordó la fruta. 
PRED:   la estudiante recuerdó la fruta. 

NÁVIRI: naku rima nemita. 
REF:    el niño recordó la flor. 
PRED:   el niño recuerdó la flor. 

NÁVIRI: kemi suni sova nemita. 
REF:    la investigadora recordó el libro en la escuela. 
PRED:   la investigadora recuerdó el libro en la escuela. 

NÁVIRI: kemi feni nemita. 
REF:    la investigadora recordó la piedra. 
PRED:   la investigadora recuerdó la piedra. 

Errores OOD: 5 / 10
chrF val: chrF2 = 100.00


## 7. Predicciones finales (test público)

**Decisión de protocolo:** para las predicciones finales se reentrena con **train + validación**, y solo en este paso. Validación se usó antes únicamente para comparar métodos, no para ajustar reglas, y esto es coherente con la regla 4 del enunciado. Debe declararse en el registro de herramientas.

La verificación `assert` comprueba que hay 40 filas y que ninguna predicción contiene el marcador `?` de token desconocido.

In [8]:
final = SlotTranslator().fit(train + val)
out = Path('/content/predicciones.csv') if Path('/content').exists() else Path('predicciones.csv')
with out.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['id', 'prediccion']); w.writeheader()
    for r in public: w.writerow({'id': r['id'], 'prediccion': final(r['naviri'])})
assert len(public) == 40
print('Guardado:', out.resolve(), '| desconocidos "?":', sum('?' in final(r['naviri']) for r in public))

Guardado: C:\Users\aml\Desktop\NLP\predicciones.csv | desconocidos "?": 0


## 8. Fase 5: nuevos mensajes

Se aplica **el mismo sistema, sin modificar ninguna regla**, a los 40 mensajes de Fase 5. Antes de exportar se hacen tres comprobaciones:

1. **Auditoría de pistas.** Se compara cada correspondencia léxica de los estudiantes (sujetos, objetos, lugares y verbos) con el léxico inducido de los datos. No hay conflictos, y dos fuentes independientes coinciden.
2. **Cobertura.** No hay palabras fuera del vocabulario ni combinaciones verbo×tiempo no vistas, así que la regla de inflexión (el punto débil de la sección 6) no interviene en Fase 5.
3. **Exportación.** Mismo formato `id,prediccion`.

**Límite de lo que podemos afirmar:** Fase 5 no tiene referencias, así que no hay una métrica medida. La confianza viene de la consistencia estructural, no de un EM calculado.

In [9]:
F5 = Path('/content/datos/evento_fase_5') if Path('/content/datos/evento_fase_5').exists() else Path('datos/evento_fase_5')
def load_utf8sig(p):
    with p.open(encoding='utf-8-sig', newline='') as f: return list(csv.DictReader(f))
f5 = load_utf8sig(F5/'LEXICON_X_Fase5_Nuevos_Mensajes.csv')
pistas = load_utf8sig(F5/'LEXICON_X_Pistas_Lexicas_Estudiantes.csv')
final = SlotTranslator().fit(train + val)
# 1) Auditoría: ¿las pistas de los estudiantes coinciden con el léxico inducido de los datos?
kind = {'sujeto': 'S', 'objeto': 'O', 'lugar': 'L'}
conflictos = []
for p in pistas:
    cat, nv, es = p['categoria'], p['naviri'], p['espanol'].strip()
    if cat in kind:
        ind = final.lex.get((kind[cat], nv))
        ind = ind[len('en '):] if (ind is None and False) else ind
        exp = es[3:] if cat == 'lugar' and es.startswith('en ') else es
        if ind is None or ind != exp: conflictos.append((nv, es, ind))
    elif cat.startswith('verbo'):
        ind = final.lex.get(('V', nv, 'PRES'))
        if ind != es: conflictos.append((nv, es, ind))
print('Conflictos pistas vs. léxico inducido:', conflictos if conflictos else 'ninguno')
# 2) Cobertura
known = {t for r in train + val for t in toks(r['naviri'])}
print('OOV en Fase 5:', sorted({w for r in f5 for w in toks(r['naviri']) if w not in known}))
seen_vt = {(k[1], k[2]) for k in final.lex if k[0] == 'V'}
print('Combos verbo×tiempo no vistos:', sum((lambda p: (p['V'], p['T']) not in seen_vt)(parse_nv(r['naviri'], final.stems)) for r in f5))
# 3) Exportar
out5 = Path('/content/predicciones_fase5.csv') if Path('/content').exists() else Path('predicciones_fase5.csv')
with out5.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['id', 'prediccion']); w.writeheader()
    for r in f5: w.writerow({'id': r['id'], 'prediccion': final(r['naviri'])})
assert len(f5) == 40 and all('?' not in final(r['naviri']) for r in f5)
print('Guardado:', out5.resolve())

Conflictos pistas vs. léxico inducido: ninguno
OOV en Fase 5: []
Combos verbo×tiempo no vistos: 0
Guardado: C:\Users\aml\Desktop\NLP\predicciones_fase5.csv


## 9. Conclusión de investigación

**Hipótesis.** H1 (composicionalidad por slots) y H2 (morfología productiva) se sostienen con la evidencia disponible.

**Evidencia cuantitativa.** EM de 0.000 con recuperación a 1.000 con gramática por slots; la curva de datos muestra ~0.97 con 80 pares; en OOD, segmentar sufijos mejora de 0.0 a 0.5.

**Ablación.** Quitar la segmentación de sufijos (un solo factor) hunde el OOD de 0.5 a 0.0. Quitar el corpus monolingüe no cambia el resultado.

**Casos fallidos.** Los cinco errores OOD son el mismo caso: verbos con cambio de raíz en español (`recordar`), que una regla de sufijo no resuelve.

**Amenazas a la validez**

- La validación estándar no discrimina: todas las combinaciones verbo×tiempo y todo el vocabulario estaban en train. Un 1.000 ahí no prueba generalización.
- El OOD usa solo 10 frases, así que la diferencia 0.5 frente a 0.0 es indicativa, no concluyente.
- El idioma es sintético: las conclusiones **no** se extrapolan a lenguas indígenas reales (regla 6 del enunciado).
- La plantilla del español (`SN [no] V SN [en SN]`) está escrita a mano; si la evaluación oculta trae estructuras distintas, el parseo fallaría y habría que inducir también esa plantilla.

**Declaración de herramientas.** Se usó un asistente de IA (Claude) para asistir en el análisis de los datos, diseñar el método, en la redacción del código y la documentación.